# Stepwise Feature Selection across Election Cycles
This notebook demonstrates how the optimal subset of predictive features varies across the 2016, 2020, and 2024 elections using AIC-based stepwise selection.

In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.simplefilter('ignore')

from bias_analysis.dataset import get_base_dataset
from bias_analysis.modeling.train import load_and_merge_features

In [5]:
def stepwise_selection(df: pd.DataFrame, target: str, candidates: list):
    included = []
    best_aic = np.inf
    raw_weights = df["total_votes"].values.astype(float)
    
    while True:
        changed = False
        
        # Forward step
        excluded = list(set(candidates) - set(included))
        new_pval = pd.Series(index=excluded, dtype=float)
        
        for new_column in excluded:
            formula = f"{target} ~ " + " + ".join(included + [new_column]) if included else f"{target} ~ {new_column}"
            try:
                model = smf.glm(formula=formula, data=df, family=sm.families.Binomial(), var_weights=raw_weights).fit(disp=0)
                new_pval[new_column] = model.aic
            except Exception:
                new_pval[new_column] = np.inf
                
        if not new_pval.empty:
            best_new_aic = new_pval.min()
            if best_new_aic < best_aic:
                best_feature = new_pval.idxmin()
                included.append(best_feature)
                best_aic = best_new_aic
                changed = True
                
        # Backward step
        if len(included) > 1:
            back_pval = pd.Series(index=included, dtype=float)
            for current_column in included:
                current_included = list(set(included) - {current_column})
                formula = f"{target} ~ " + " + ".join(current_included) if current_included else f"{target} ~ 1"
                try:
                    model = smf.glm(formula=formula, data=df, family=sm.families.Binomial(), var_weights=raw_weights).fit(disp=0)
                    back_pval[current_column] = model.aic
                except Exception:
                    back_pval[current_column] = np.inf
                    
            best_back_aic = back_pval.min()
            if best_back_aic < best_aic:
                worst_feature = back_pval.idxmin()
                included.remove(worst_feature)
                best_aic = best_back_aic
                changed = True
                
        if not changed:
            break
            
    final_formula = f"{target} ~ " + " + ".join(included) if included else f"{target} ~ 1"
    dropped = list(set(candidates) - set(included))
    return final_formula, dropped

In [6]:
candidates = [
    "audience_mean_partisanship", "author_partisanship",
    "candidate_order", "formality_bias", "positive_ideology_score",
    "undirected_sentiment", "sentiment_intensity", "toxicity_score",
    "bias_confirmation", "bias_anchoring", "bias_availability",
    "bias_social_desirability", "bias_acquiescence", "bias_demand_characteristics",
    "author_gender_male", "author_age_30_39", "author_age_40_over"
]

for election in ["us16", "us20", "us24"]:
    print(f"\n{'='*50}\nAnalyzing Election: {election}\n{'='*50}")
    
    base_df = get_base_dataset(election=election)
    df = load_and_merge_features(base_df, election=election)
        
    for col in candidates:
        if col in df.columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(0.0)
    
    analysis_df = df.dropna(subset=["positive_share"]).copy()
    analysis_df["positive_share"] = analysis_df["positive_share"].clip(0.0, 1.0)
    analysis_df["total_votes"] = analysis_df["total_votes"].clip(lower=1)
    
    formula, dropped = stepwise_selection(analysis_df, "positive_share", candidates)
    
    print(f"Features Dropped by Stepwise:\n  {dropped if dropped else 'None (All 17 features selected!)'}")
    print(f"\nFinal Optimal Formula for {election}:\n  {formula}")


Analyzing Election: us16
2026-06-12 15:43:44.996 | INFO     | bias_analysis.dataset:get_base_dataset:354 - Loading cached base dataset from C:\Users\utente\Desktop\Work\TweetPollBias\data\processed\us16\base_dataset_cache.pkl
2026-06-12 15:43:45.045 | WARNING  | bias_analysis.modeling.train:load_and_merge_features:222 - Dropped 3 rows with residual NaN/Inf values
2026-06-12 15:43:45.045 | INFO     | bias_analysis.modeling.train:load_and_merge_features:224 - Merged dataset: 758 polls, 47 columns
Features Dropped by Stepwise:
  ['bias_acquiescence', 'bias_confirmation']

Final Optimal Formula for us16:
  positive_share ~ audience_mean_partisanship + formality_bias + candidate_order + bias_demand_characteristics + author_age_40_over + author_partisanship + undirected_sentiment + sentiment_intensity + positive_ideology_score + author_age_30_39 + toxicity_score + author_gender_male + bias_anchoring + bias_social_desirability + bias_availability

Analyzing Election: us20
2026-06-12 15:43:48